# Sample Payload Real Flow Test

This notebook keeps the flow simple:
- load the sample payload
- run real issue categorization
- prove real GitHub fetch works with your token
- generate the BOM-friendly remediation plan
- push the new branch and PR

It does not use `TestClient` or `EventStore`.

In [1]:
import json
import sys
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root from the current working directory.")


ROOT = find_repo_root(Path.cwd())
PACKAGE_ROOT = ROOT / "dev_test" / "jenkins_webhook_langgraph_flow" / "python"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

from webhook_langgraph_flow import workflow, github_service

In [2]:
PAYLOAD_PATH = ROOT / "dev_test" / "jenkins_webhook_langgraph_flow" / "payloads" / "sample_webhook_payload.json"
payload = json.loads(PAYLOAD_PATH.read_text(encoding="utf-8"))

# Optional: add one explicit non-vulnerability failure line so you can see OTHER=1 too.
INJECT_OTHER_ISSUE = True
if INJECT_OTHER_ISSUE and "ERROR Deployment blocked." not in payload["console_output"]:
    payload["console_output"] += "\nERROR Deployment blocked."

payload

{'job_name': 'spring-boot-backend',
 'build_number': 10,
 'build_url': 'http://localhost:8080/job/spring-boot-backend/10/',
 'repo': 'nishathmnha/demo-springboot-vuln-service',
 'branch': 'main',
 'target_branch': 'main',
 'status': 'FAILED',
 'console_output': 'jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml (pom)\n│                  Library                  │  Vulnerability   │ Severity │ Status │ Installed Version │        Fixed Version        │                            Title                             │\n│ org.apache.logging.log4j:log4j-core       │ CVE-2021-44228   │ CRITICAL │ fixed  │ 2.14.1            │ 2.17.1                      │ log4j-core remote code execution                             │\nERROR Deployment blocked.',
 'dependency_file_paths': ['jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml'],
 'docker_file_paths': ['jenkins-webhook-and-github-setup/demo-springboot-vuln-service/Dockerfile']}

## Configuration

This payload does not include `local_repo_path`, so the workflow will fetch from GitHub.

In [3]:
payload["repo"] = "nishathmnha/demo-springboot-vuln-service"
payload["branch"] = "main"
payload["target_branch"] = "main"
payload.pop("local_repo_path", None)

# Set False if you want to stop before creating a real branch and PR.
RUN_REAL_GITHUB_PUSH = True

payload

{'job_name': 'spring-boot-backend',
 'build_number': 10,
 'build_url': 'http://localhost:8080/job/spring-boot-backend/10/',
 'repo': 'nishathmnha/demo-springboot-vuln-service',
 'branch': 'main',
 'target_branch': 'main',
 'status': 'FAILED',
 'console_output': 'jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml (pom)\n│                  Library                  │  Vulnerability   │ Severity │ Status │ Installed Version │        Fixed Version        │                            Title                             │\n│ org.apache.logging.log4j:log4j-core       │ CVE-2021-44228   │ CRITICAL │ fixed  │ 2.14.1            │ 2.17.1                      │ log4j-core remote code execution                             │\nERROR Deployment blocked.',
 'dependency_file_paths': ['jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml'],
 'docker_file_paths': ['jenkins-webhook-and-github-setup/demo-springboot-vuln-service/Dockerfile']}

In [4]:
github_token = github_service.GITHUB_TOKEN.strip()
old_github_dry_run = github_service.GITHUB_DRY_RUN
github_service.GITHUB_DRY_RUN = not RUN_REAL_GITHUB_PUSH

print("GitHub token configured:", bool(github_token))
print("GitHub dry-run enabled:", github_service.GITHUB_DRY_RUN)

if not github_token:
    raise RuntimeError("GitHub token was not found in .env. Set PAYLOAD_MONITOR_GITHUB_TOKEN or GITHUB_TOKEN.")

GitHub token configured: True
GitHub dry-run enabled: False


## 1. Categorize issues as VA / OTHER

In [5]:
analysis = workflow.run_analysis(payload, event_id="nb-real-analysis")
pprint(analysis, width=140, sort_dicts=False)

{'event_id': 'nb-real-analysis',
 'created_at': '2026-05-01T15:21:39.437955+00:00',
 'repo': 'nishathmnha/demo-springboot-vuln-service',
 'branch': 'main',
 'file_paths': ['jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml',
                'jenkins-webhook-and-github-setup/demo-springboot-vuln-service/Dockerfile'],
 'issues': [{'issue_category': 'VA',
             'package_name': 'org.apache.logging.log4j:log4j-core',
             'target_file_hint': 'jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml',
             'installed_version': '2.14.1',
             'fixed_versions': ['2.17.1'],
             'cve_ids': ['CVE-2021-44228'],
             'detailed_issue': 'org.apache.logging.log4j:log4j-core in '
                               'jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml is vulnerable.',
             'initial_fix': 'Prefer the parent/BOM/property owner over scattered direct overrides.'},
            {'issue_catego

## 2. Real GitHub fetch

This proves the exact dependency and Docker file paths can be fetched from the real repo.

In [6]:
fetch_paths = payload["dependency_file_paths"] + payload["docker_file_paths"]
fetched_files = github_service.fetch_repo_files_by_paths(payload["repo"], payload["branch"], fetch_paths)
pprint({
    "repo": payload["repo"],
    "branch": payload["branch"],
    "paths": list(fetched_files.keys()),
    "previews": {path: content[:200] for path, content in fetched_files.items()},
}, width=160, sort_dicts=False)

{'repo': 'nishathmnha/demo-springboot-vuln-service',
 'branch': 'main',
 'paths': ['jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml', 'jenkins-webhook-and-github-setup/demo-springboot-vuln-service/Dockerfile'],
 'previews': {'jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml': '<?xml version="1.0" encoding="UTF-8"?>\n'
                                                                                       '<project xmlns="http://maven.apache.org/POM/4.0.0"\n'
                                                                                       '         xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"\n'
                                                                                       '         xsi:schemaLocation="http://maven.apach',
              'jenkins-webhook-and-github-setup/demo-springboot-vuln-service/Dockerfile': 'FROM maven:3.9.9-eclipse-temurin-17 AS build\n'
                                                                

## 3. Generate BOM-friendly plan

This uses the same real payload and fetches from GitHub because `local_repo_path` is absent.

In [7]:
proposal = workflow.run_prepare_fix(payload, event_id="nb-real-prepare")
pprint({
    "event_id": proposal["event_id"],
    "repo": proposal["repo"],
    "branch": proposal["branch"],
    "file_paths": proposal["file_paths"],
    "resolved_paths": proposal["resolved_paths"],
    "branch_name": proposal["branch_name"],
    "plan_notes": proposal["plan"].get("notes", []),
    "planned_files": [item["path"] for item in proposal["plan"].get("file_changes", [])],
}, width=160, sort_dicts=False)

{'event_id': 'nb-real-prepare',
 'repo': 'nishathmnha/demo-springboot-vuln-service',
 'branch': 'main',
 'file_paths': ['jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml',
                'jenkins-webhook-and-github-setup/demo-springboot-vuln-service/Dockerfile'],
 'resolved_paths': [{'issue_index': 0, 'path': 'jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml'},
                    {'issue_index': 1, 'path': 'jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml'}],
 'branch_name': 'ai-va-fix/20260501152150',
 'plan_notes': [],
 'planned_files': ['jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml']}


In [8]:
if proposal["plan"].get("file_changes"):
    first_change = proposal["plan"]["file_changes"][0]
    print(first_change["path"])
    print("\n".join(first_change["diff_preview"]))
else:
    print("No file changes were planned.")

jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml
-            <version>2.14.1</version>
+            <version>2.17.1</version>


## 4. Push the new branch

This creates the real branch and PR when `RUN_REAL_GITHUB_PUSH = True`.

In [9]:
apply_result = workflow.run_apply_fix(
    payload,
    event_id="nb-real-apply",
    push_enabled=True,
)
pprint(apply_result, width=160, sort_dicts=False)

{'event_id': 'nb-real-apply',
 'created_at': '2026-05-01T15:22:03.672394+00:00',
 'branch_name': 'ai-va-fix/20260501152157',
 'plan': {'file_changes': [{'path': 'jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml',
                            'updated_content': '<?xml version="1.0" encoding="UTF-8"?>\n'
                                               '<project xmlns="http://maven.apache.org/POM/4.0.0"\n'
                                               '         xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"\n'
                                               '         xsi:schemaLocation="http://maven.apache.org/POM/4.0.0 https://maven.apache.org/xsd/maven-4.0.0.xsd">\n'
                                               '    <modelVersion>4.0.0</modelVersion>\n'
                                               '\n'
                                               '    <parent>\n'
                                               '        <groupId>org.springframework.boot</gr

## 5. Assertions

In [10]:
assert analysis["issue_counts"]["VA"] >= 1
assert analysis["issues"][0]["issue_category"] == "VA"

if INJECT_OTHER_ISSUE:
    assert analysis["issue_counts"]["OTHER"] >= 1

assert list(fetched_files.keys()) == [
    "jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml",
    "jenkins-webhook-and-github-setup/demo-springboot-vuln-service/Dockerfile",
]
assert proposal["file_paths"] == [
    "jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml",
    "jenkins-webhook-and-github-setup/demo-springboot-vuln-service/Dockerfile",
]
assert proposal["plan"]["file_changes"][0]["path"] == "jenkins-webhook-and-github-setup/demo-springboot-vuln-service/pom.xml"
assert "2.17.1" in proposal["plan"]["file_changes"][0]["updated_content"]
assert apply_result["push_result"]["status"] in {"dry-run", "pushed"}

if RUN_REAL_GITHUB_PUSH:
    assert apply_result["push_result"]["status"] == "pushed"
    assert apply_result["push_result"].get("pull_request_url")

print("Real workflow checks passed.")

Real workflow checks passed.


In [ ]:
github_service.GITHUB_DRY_RUN = old_github_dry_run